# 08 モデルの改善：テストの年での評価

`07_improve` で選んだ設定のモデル（`models/cloud_base.joblib`）を、**テストの年（2025）** で評価し、
成功基準（`docs/spec.md`「9. 分割と評価」）を判定する。前のモデル（v1。`04_train`）と並べる。結果は `reports/metrics_improved.json` に保存する。

- **テストの年を見るのは 2 回目**（1 回目は `05_evaluate`、v1）。設定は `07_improve` で、検証の年までだけを使って決めてある。ここでは何も選ばない。
- モデルはここでは学習しない（読み込むだけ）。`07_improve` で設定が選ばれなかった場合は、このノートブックは実行しない（エラーで止まる）。
- 採る条件（`ml/improve.py` の `adopt`。結果を見る前に決めた）：2 都市とも、成功基準の 1・2 を両方満たす。
  1. 全体の MAE が、`always_cap` より 10% 以上小さい。
  2. 上限より下の行だけの MAE が、`rule_of_thumb` より 10% 以上小さい。
- 参考として、検証の年（2024）の指標も同じ形で並べる（設定を選ぶのに使った年なので、良く出やすい）。
- 既定は **サンプル**（`tests/fixtures/data_raw/` の 2 日ぶん。その場で小さく学習して、動くことだけを確かめる。数値に意味はない。一時的な場所に書く）。
  コミットしてある `reports/metrics_improved.json` は、環境変数 `DATA_FETCH_MODE=full` を付けて全期間で実行したもの。外部とは通信しない。
- 入力は Open-Meteo の過去データ（再解析）。API が使うのは予報なので、実際の精度はここでの数値より悪い可能性がある（`docs/model-card.md`）。

出典：Weather data by Open-Meteo.com（CC BY 4.0）、Iowa Environmental Mesonet（Iowa State University）。

In [ ]:
import json
import os
import sys
import tempfile
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from app.cities import City
from ml import (
    archive,
    baseline,
    dataset,
    evaluate,
    features,
    improve,
    metar,
    model,
    raw_store,
    split,
    target,
)

## 0. 設定・読み込み

In [ ]:
MODE = os.environ.get("DATA_FETCH_MODE", "sample")
# 新しいモデル（07 が保存したもの）と、前のモデル（v1）。
NEW_MODEL_FILE = "cloud_base.joblib"
PREVIOUS_MODEL_FILE = "cloud_base_v1.joblib"

if MODE == "full":
    OUTPUT = ROOT / "reports" / "metrics_improved.json"
    SPLITS = {name: list(split.SPLIT_YEARS[name]) for name in (split.VALIDATION, split.TEST)}
    CRITERIA_SPLIT = split.TEST
elif MODE == "sample":
    OUTPUT = Path(tempfile.mkdtemp()) / "metrics_improved.json"
    # fixture は 2025 年の 2 日ぶんだけなので、分割で絞らずに 1 つの組として評価する。
    SPLITS = {"sample": [2025]}
    CRITERIA_SPLIT = "sample"
else:
    raise ValueError("DATA_FETCH_MODE must be 'sample' or 'full'")

pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", "{:.2f}".format)
print(f"mode={MODE}", list(SPLITS))

In [ ]:
if MODE == "full":
    observations = dataset.load_dataset(ROOT / "data" / "processed" / "observations.parquet")
else:
    raw_root = ROOT / "tests" / "fixtures" / "data_raw"
    parts = {}
    for city in City:
        weather_path = raw_store.raw_path(raw_root, "open_meteo_archive", city, 2025, "json")
        metar_path = raw_store.raw_path(raw_root, "iem", city, 2025, "csv")
        parts[city.value] = (
            archive.parse_archive(weather_path.read_bytes().decode("utf-8")),
            metar.parse_metar(metar_path.read_bytes().decode("utf-8")),
        )
    observations = dataset.build_dataset(parts)

dataset.validate_dataset(observations)
X = features.features(observations)
y = target.target_m(observations)
observations["split"] = split.split(observations) if MODE == "full" else "sample"
print(observations.shape, observations["time"].min(), observations["time"].max())

In [ ]:
if MODE == "full":
    improvement = json.loads((ROOT / "reports" / "improvement.json").read_text(encoding="utf-8"))
    setting = improvement["selected"]
    if setting is None:
        # 選ぶ決まりの 4：選ばれた設定がなければ、テストの年は評価しない。
        raise RuntimeError("no setting was selected; the test year is not evaluated")
    new_model = model.load(ROOT / "models" / NEW_MODEL_FILE)
    previous_model = model.load(ROOT / "models" / PREVIOUS_MODEL_FILE)
else:
    # 07 の出力は読まずに、同じ行でその場で学習する（動くことの確認だけ）。
    setting = improve.SELECTABLE[0]
    candidates = improve.fit_candidates(X, y, X, y, (2025,))
    new_model, previous_model = candidates[setting], candidates[improve.REFERENCE]

assert new_model.model_version == model.MODEL_VERSION_V2
assert previous_model.model_version == model.MODEL_VERSION_V1
print(setting, new_model.model_version, new_model.train_years, previous_model.model_version)

## 1. 予測

In [ ]:
baselines = {
    "rule_of_thumb": baseline.baseline_m(
        observations["temperature_c"], observations["dew_point_c"]
    ),
    "always_cap": pd.Series(target.CEILING_CAP_M, index=observations.index, dtype="float64"),
}
predictions = {
    "model": model.predict_m(new_model, X),
    "previous_model": model.predict_m(previous_model, X),
    **baselines,
}
pd.DataFrame(predictions).describe().T

## 2. 評価

全体・都市別・都市 × 季節別（季節は現地の月）。

In [ ]:
results = {}
for name, years in SPLITS.items():
    rows = observations["split"] == name
    results[name] = {
        "years": years,
        "predictions": {
            label: evaluate.report(observations[rows], y[rows], prediction[rows])
            for label, prediction in predictions.items()
        },
    }

pd.DataFrame(
    [
        {"split": name, "prediction": label, "city": city, **metrics}
        for name, result in results.items()
        for label, reports in result["predictions"].items()
        for city, metrics in reports["by_city"].items()
    ]
)

## 3. 成功基準と、採るかどうか

テストの年で、都市ごとに判定する。判定は丸める前の値で行う（ちょうど 10% は満たす）。

In [ ]:
def criteria_for(rows, prediction):
    """`rows` の行の、都市ごとの成功基準の判定（丸める前の値）。"""
    result = {}
    for city in sorted(observations["city"][rows].unique()):
        city_rows = rows & (observations["city"] == city)
        result[city] = evaluate.success_criteria(
            model=evaluate.evaluate(y[city_rows], prediction[city_rows]),
            always_cap=evaluate.evaluate(y[city_rows], baselines["always_cap"][city_rows]),
            baseline=evaluate.evaluate(y[city_rows], baselines["rule_of_thumb"][city_rows]),
        )
    return result


def rounded(criteria):
    """`reports/` に書く形（割合は小数 4 桁）。"""
    return {
        city: {
            key: round(value, 4) if isinstance(value, float) else value
            for key, value in result.items()
        }
        for city, result in criteria.items()
    }


criteria_rows = observations["split"] == CRITERIA_SPLIT
success_criteria = criteria_for(criteria_rows, predictions["model"])
previous_success_criteria = criteria_for(criteria_rows, predictions["previous_model"])
adopted = improve.adopt(success_criteria)

print("adopted:", adopted)
pd.concat(
    {
        "model": pd.DataFrame(rounded(success_criteria)).T,
        "previous_model": pd.DataFrame(rounded(previous_success_criteria)).T,
    }
)

## 4. 保存

実行した時刻は入れない（再実行しても同じファイルになる）。

In [ ]:
report = {
    "mode": MODE,
    "model_version": new_model.model_version,
    "previous_model_version": previous_model.model_version,
    "setting": setting,
    "train_years": list(new_model.train_years),
    "cap_m": target.CEILING_CAP_M,
    # 2025 年は、05_evaluate（v1）で 1 回見ている。
    "test_year_evaluation": "second",
    "predictions": {
        "model": "LightGBM with the setting selected in 07_improve, clipped to [0, cap_m]",
        "previous_model": "LightGBM of 04_train (v1), clipped to [0, cap_m]",
        "rule_of_thumb": "baseline: 125 * (temperature_c - dew_point_c) with Open-Meteo values",
        "always_cap": "reference: always cap_m",
    },
    "success_criteria_split": CRITERIA_SPLIT,
    "success_criteria_min_improvement": evaluate.SUCCESS_MIN_IMPROVEMENT,
    "success_criteria": rounded(success_criteria),
    "previous_success_criteria": rounded(previous_success_criteria),
    "adopted": adopted,
    "splits": results,
}

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
text = json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False) + "\n"
OUTPUT.write_text(text, encoding="utf-8", newline="\n")
assert json.loads(OUTPUT.read_text(encoding="utf-8")) == report
print(OUTPUT.name, len(text))